# 04. Análisis estadístico y exploratorio — Avance 03

**Proyecto A&C — Análisis de accidentes y hechos de tránsito en la Ciudad de México**  
**Equipo:** Manuel Soto Romero y Araceli Liliana Reyes Cabello · UACM · Inteligencia de Negocios · Grupo 313 · 2026-2

## Objetivo
Generar estadísticas descriptivas y gráficas a partir de los resultados limpios de SSC. Los análisis se realizan únicamente cuando se identifica una variable compatible. No se inventan columnas ni se confunde el número de hechos con el número de personas lesionadas.

**Pregunta de negocio:** ¿En qué alcaldías y franjas horarias de la Ciudad de México se registra la mayor cantidad de personas lesionadas en hechos de tránsito, de acuerdo con los registros disponibles a partir de 2018?

**Precaución metodológica:** las tablas SSC tienen granularidades distintas. No se suman ni se unen automáticamente. C5 e INEGI-ATUS tampoco se agregan a SSC; cada fuente requiere un análisis independiente.

## 1. Entorno y selección de la ejecución SSC

Busco carpetas de ejecución reales, excluyo `.ipynb_checkpoints` y acepto tanto CSV normales como CSV comprimidos. La selección se muestra antes de calcular resultados.

In [ ]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import unicodedata

TZ = ZoneInfo("America/Mexico_City")

def localizar_raiz_proyecto():
    actual = Path.cwd().resolve()
    candidatos = [actual, *actual.parents]

    for carpeta in candidatos:
        if carpeta.name.lower() == "avance03":
            return carpeta

    for carpeta in candidatos:
        if (carpeta / "data" / "processed").exists():
            return carpeta

    raise FileNotFoundError(
        "No se pudo localizar la carpeta Avance03. "
        "Abre Jupyter desde la carpeta del proyecto."
    )

ROOT = localizar_raiz_proyecto()

# Carpetas de salida de Avance03
SSC_OUT = ROOT / "data" / "processed" / "SSC"
ANALISIS_ROOT = ROOT / "data" / "processed" / "Analisis"

RUN_ID = datetime.now(TZ).strftime("%Y%m%d_%H%M%S")
RUN = ANALISIS_ROOT / RUN_ID
RUN.mkdir(parents=True, exist_ok=True)

# Archivos existentes de la etapa 3 del Avance 02
PROYECTO = ROOT.parent.parent
SSC_ETAPA3 = (
    PROYECTO / "resultados" / "resultados_SSC" / "etapa3"
)

archivos_esperados = {
    "HechosTransito": "HechosTransito_SSC_etapa3.csv",
    "VehiculosInvolucrados": "VehiculosInvolucrados_SSC_etapa3.csv",
    "PersonasLyFporSexo": "PersonasLyFporSexo_SSC_etapa3.csv",
    "PersonasLyFporEdad": "PersonasLyFporEdad_SSC_etapa3.csv",
    "PersonasLyFporTipoPersona": "PersonasLyFporTipoPersona_SSC_etapa3.csv"
}

candidatos_run = []

# 1. Buscar primero una ejecución completa de Avance03
if SSC_OUT.exists():
    runs = sorted(
        [
            p for p in SSC_OUT.iterdir()
            if p.is_dir() and not p.name.startswith(".")
        ],
        key=lambda p: p.name,
        reverse=True
    )

    conjuntos_esperados = [
        "hechostransito",
        "vehiculosinvolucrados",
        "personaslyfporsexo",
        "personaslyfporedad",
        "personaslyfportipopersona"
    ]

    for carpeta in runs:
        archivos_run = sorted(
            list(carpeta.glob("*_limpio.csv")) +
            list(carpeta.glob("*_limpio.csv.gz"))
        )

        nombres = [
            re.sub(r"[^a-z0-9]+", "", p.name.lower())
            for p in archivos_run
        ]

        faltantes = [
            token for token in conjuntos_esperados
            if not any(token in nombre for nombre in nombres)
        ]

        if not faltantes:
            candidatos_run.append((carpeta, archivos_run))

# 2. Si no existe una ejecución completa, usar etapa3
if candidatos_run:
    SSC_RUN, archivos = candidatos_run[0]
    ORIGEN_SSC = "Avance03 - ETL SSC"

else:
    if not SSC_ETAPA3.exists():
        raise FileNotFoundError(
            f"No se encontró la carpeta de datos SSC: {SSC_ETAPA3}"
        )

    faltantes = [
        nombre for nombre in archivos_esperados.values()
        if not (SSC_ETAPA3 / nombre).exists()
    ]

    if faltantes:
        raise FileNotFoundError(
            "Faltan archivos en la carpeta SSC etapa3:\n" +
            "\n".join(faltantes) +
            f"\nRuta revisada: {SSC_ETAPA3}"
        )

    archivos = [
        SSC_ETAPA3 / nombre
        for nombre in archivos_esperados.values()
    ]

    SSC_RUN = SSC_ETAPA3
    ORIGEN_SSC = "Avance02 - resultados etapa3"

print("Origen de datos SSC:", ORIGEN_SSC)
print("Carpeta SSC seleccionada:", SSC_RUN)

print("\nArchivos encontrados:")
for archivo in archivos:
    print(
        " -", archivo.name,
        f"({archivo.stat().st_size / (1024**2):.2f} MB)"
    )

print("\nCarpeta de resultados del análisis:", RUN)


## 2. Funciones de lectura y detección de variables

Los archivos comprimidos se leen con `compression="infer"`. Si una columna no se encuentra, el análisis correspondiente se marca como pendiente en lugar de provocar un `KeyError`.

In [ ]:
def nrm(valor):
    texto = unicodedata.normalize("NFKD", str(valor))
    texto = texto.encode("ascii", "ignore").decode("ascii").lower()
    return re.sub(r"[^a-z0-9]+", "_", texto).strip("_")


def findcol(df, aliases):
    mapa = {nrm(col): col for col in df.columns}
    for alias in aliases:
        if nrm(alias) in mapa:
            return mapa[nrm(alias)]
    return None


def leer_resultado(path):
    return pd.read_csv(
        path, dtype="string", encoding="utf-8-sig",
        compression="infer", low_memory=False
    )


def guardar_resumen(df, nombre):
    destino = RUN / nombre
    df.to_csv(destino, index=False, encoding="utf-8-sig", compression="gzip")
    return destino

# Cargo cada archivo limpio por separado; no concateno tablas de distinta granularidad.
tablas_ssc = {}
for archivo in archivos:
    try:
        tablas_ssc[archivo.name] = leer_resultado(archivo)
        print(f"Leído: {archivo.name} | {len(tablas_ssc[archivo.name]):,} filas")
    except Exception as error:
        print(f"AVISO: no se pudo leer {archivo.name}: {error}")

if not tablas_ssc:
    raise ValueError(
        "No se pudo leer ningún CSV limpio. Revisa si la última ejecución SSC quedó incompleta."
    )

## 3. Identificar tabla de hechos y variables disponibles

No supongo que la tabla de hechos contenga un conteo de personas lesionadas. Primero identifico los campos y muestro las columnas detectadas.

In [ ]:
candidatos_hechos = [
    nombre for nombre in tablas_ssc
    if "hecho" in nrm(nombre) or "transito" in nrm(nombre)
]

if candidatos_hechos:
    nombre_hechos = candidatos_hechos[0]
    hechos = tablas_ssc[nombre_hechos].copy()
    print("Tabla seleccionada:", nombre_hechos)
    print("Dimensiones:", hechos.shape)
    print("Columnas:", list(hechos.columns))
else:
    hechos = None
    print("No se identificó una tabla de hechos por su nombre. Se omiten análisis por hecho.")

COL_ALCALDIA = None
COL_LESIONADOS = None
COL_FECHA = None
COL_HORA = None

if hechos is not None:
    COL_ALCALDIA = findcol(hechos, ["alcaldia", "alcaldía", "nombre_alcaldia"])
    COL_LESIONADOS = findcol(hechos, [
        "personas_lesionadas", "lesionados", "total_lesionados",
        "numero_lesionados", "num_lesionados", "personas_heridas"
    ])
    COL_FECHA = findcol(hechos, ["fecha_ocurrencia", "fecha_evento", "fecha_hecho", "fecha"])
    COL_HORA = findcol(hechos, ["hora_ocurrencia", "hora_evento", "hora_hecho", "hora_accidente"])

print({
    "alcaldia": COL_ALCALDIA,
    "personas_lesionadas": COL_LESIONADOS,
    "fecha_ocurrencia": COL_FECHA,
    "hora_ocurrencia": COL_HORA
})

## 4. Distribución territorial

Si existe una variable explícita de personas lesionadas, calculo el total por alcaldía. Si no existe, no sustituyo el conteo de lesionados por el número de filas; únicamente puedo presentar un conteo separado de hechos cuando la alcaldía está disponible.

In [ ]:
if hechos is not None and COL_ALCALDIA:
    if COL_LESIONADOS:
        lesionados = pd.to_numeric(hechos[COL_LESIONADOS], errors="coerce")
        validos = hechos.loc[lesionados.notna() & lesionados.ge(0)].copy()
        validos["_lesionados_num"] = lesionados.loc[validos.index]
        validos["_alcaldia"] = validos[COL_ALCALDIA].astype("string").str.strip()
        por_alcaldia = (
            validos.loc[validos["_alcaldia"].notna() & validos["_alcaldia"].ne("")]
            .groupby("_alcaldia")["_lesionados_num"]
            .sum().sort_values(ascending=False)
            .rename("personas_lesionadas").reset_index()
        )
        total = por_alcaldia["personas_lesionadas"].sum()
        por_alcaldia["porcentaje"] = (
            por_alcaldia["personas_lesionadas"] / total * 100 if total else np.nan
        )
        display(por_alcaldia)
        guardar_resumen(por_alcaldia, "lesionados_por_alcaldia.csv.gz")
        if len(por_alcaldia):
            ax = por_alcaldia.sort_values("personas_lesionadas").plot.barh(
                x="_alcaldia", y="personas_lesionadas", legend=False, figsize=(10, 7)
            )
            ax.set_title("Personas lesionadas por alcaldía — SSC")
            ax.set_xlabel("Personas lesionadas")
            ax.set_ylabel("Alcaldía")
            plt.tight_layout()
            plt.show()
    else:
        conteo_hechos = (
            hechos.assign(_alcaldia=hechos[COL_ALCALDIA].astype("string").str.strip())
            .loc[lambda x: x["_alcaldia"].notna() & x["_alcaldia"].ne("")]
            .groupby("_alcaldia").size().rename("numero_de_hechos")
            .sort_values(ascending=False).reset_index()
        )
        print(
            "No se encontró una columna de personas lesionadas. "
            "El siguiente resultado cuenta hechos, NO personas lesionadas."
        )
        display(conteo_hechos)
        guardar_resumen(conteo_hechos, "hechos_por_alcaldia_no_lesionados.csv.gz")
else:
    print("Pendiente: no se identificó tabla de hechos o columna de alcaldía.")

## 5. Distribución por franja horaria

Utilizo exclusivamente la hora de ocurrencia. Los intervalos son madrugada 00:00–05:59, mañana 06:00–11:59, tarde 12:00–17:59 y noche 18:00–23:59. No sustituyo una hora ausente por la hora de creación o cierre.

In [ ]:
def extraer_hora(serie):
    texto = serie.astype("string").str.strip()
    # Primero intento formatos HH:MM o HH:MM:SS.
    capturada = texto.str.extract(r"^([0-2]?\d)(?::[0-5]\d)?(?::[0-5]\d)?$", expand=False)
    hora = pd.to_numeric(capturada, errors="coerce")
    # Alternativa para formatos reconocibles por pandas (por ejemplo, AM/PM).
    faltantes = hora.isna()
    if faltantes.any():
        parseada = pd.to_datetime(texto.loc[faltantes], errors="coerce")
        hora.loc[faltantes] = parseada.dt.hour
    return hora.where(hora.between(0, 23))


if hechos is not None and COL_HORA:
    horas = extraer_hora(hechos[COL_HORA])
    franjas = pd.cut(
        horas, bins=[-1, 5, 11, 17, 23],
        labels=["Madrugada", "Mañana", "Tarde", "Noche"]
    )
    if COL_LESIONADOS:
        conteo = pd.to_numeric(hechos[COL_LESIONADOS], errors="coerce")
        temporal = pd.DataFrame({"franja": franjas, "lesionados": conteo})
        por_franja = (
            temporal.dropna(subset=["franja", "lesionados"])
            .loc[lambda x: x["lesionados"].ge(0)]
            .groupby("franja", observed=False)["lesionados"]
            .sum().rename("personas_lesionadas").reset_index()
        )
        display(por_franja)
        guardar_resumen(por_franja, "lesionados_por_franja_horaria.csv.gz")
        if len(por_franja):
            por_franja.plot.bar(
                x="franja", y="personas_lesionadas", legend=False,
                title="Personas lesionadas por franja de ocurrencia", rot=0
            )
            plt.ylabel("Personas lesionadas")
            plt.xlabel("Franja horaria")
            plt.tight_layout()
            plt.show()
    else:
        por_franja = franjas.value_counts(sort=False, dropna=True).rename_axis(
            "franja"
        ).reset_index(name="numero_de_hechos")
        print("No hay conteo de lesionados; esta tabla representa hechos, no personas.")
        display(por_franja)
        guardar_resumen(por_franja, "hechos_por_franja_no_lesionados.csv.gz")
else:
    print("No se identificó hora de ocurrencia; se omite el análisis por franja.")

## 6. Estadística descriptiva de personas lesionadas por hecho

Media, mediana, moda, desviación estándar y cuartiles describen el conteo de lesionados por registro de hecho, no el riesgo individual ni el total de incidentes. Se calculan únicamente si existe una variable numérica compatible.

In [ ]:
if hechos is not None and COL_LESIONADOS:
    serie = pd.to_numeric(hechos[COL_LESIONADOS], errors="coerce")
    serie = serie[serie.notna() & serie.ge(0)]
    if len(serie):
        modas = serie.mode().tolist()
        estadisticas = pd.DataFrame([
            {"medida": "Media", "valor": serie.mean()},
            {"medida": "Mediana", "valor": serie.median()},
            {"medida": "Moda(s)", "valor": ", ".join(map(str, modas))},
            {"medida": "Desviación estándar", "valor": serie.std()},
            {"medida": "Q1 (25%)", "valor": serie.quantile(.25)},
            {"medida": "Q2 (50%)", "valor": serie.quantile(.50)},
            {"medida": "Q3 (75%)", "valor": serie.quantile(.75)},
            {"medida": "Mínimo", "valor": serie.min()},
            {"medida": "Máximo", "valor": serie.max()},
            {"medida": "N válido", "valor": int(serie.count())}
        ])
        display(estadisticas)
        guardar_resumen(estadisticas, "estadisticas_descriptivas_lesionados.csv.gz")

        plt.figure(figsize=(9, 5))
        plt.boxplot(serie, vert=True, showmeans=True)
        plt.title("Caja y bigotes — lesionados por registro de hecho (SSC)")
        plt.ylabel("Personas lesionadas por hecho")
        plt.grid(axis="y", alpha=.25)
        plt.tight_layout()
        plt.show()
    else:
        print("La columna existe, pero no contiene conteos numéricos no negativos.")
else:
    print("Estadística de lesionados pendiente: falta una columna identificable de lesionados.")

## 7. Análisis complementario de edades SSC

La edad se analiza desde la tabla de personas por edad, no desde la tabla de hechos. Si la tabla está agregada por edad, se utiliza una columna de frecuencia cuando existe. No se trata cada fila agregada como si representara una sola persona.

In [ ]:
archivos_edad = [
    (nombre, df) for nombre, df in tablas_ssc.items()
    if "edad" in nrm(nombre)
]

if archivos_edad:
    nombre_edad, personas_edad = archivos_edad[0]
    COL_EDAD = findcol(personas_edad, ["edad", "edad_persona", "anios", "años"])
    COL_FRECUENCIA = findcol(personas_edad, [
        "total", "cantidad", "personas", "frecuencia", "conteo", "numero_personas"
    ])

    if COL_EDAD:
        edades = pd.to_numeric(personas_edad[COL_EDAD], errors="coerce")
        mascara_edad = edades.between(0, 99)
        edades = edades.loc[mascara_edad]

        if COL_FRECUENCIA:
            frecuencias = pd.to_numeric(
                personas_edad.loc[mascara_edad, COL_FRECUENCIA], errors="coerce"
            ).fillna(0)
            validos = frecuencias.gt(0)
            edades = edades.loc[validos]
            frecuencias = frecuencias.loc[validos]
            if len(edades):
                total_personas = frecuencias.sum()
                media = np.average(edades, weights=frecuencias)
                varianza = np.average((edades - media) ** 2, weights=frecuencias)
                orden = np.argsort(edades.to_numpy())
                edades_ord = edades.to_numpy()[orden]
                frec_ord = frecuencias.to_numpy()[orden]
                acumulada = np.cumsum(frec_ord)
                def cuant_peso(q):
                    return float(edades_ord[np.searchsorted(acumulada, q * total_personas, side="left")])
                moda = float(edades.loc[frecuencias.idxmax()])
                resumen_edad = pd.DataFrame([
                    {"medida": "Media ponderada", "valor": media},
                    {"medida": "Mediana ponderada", "valor": cuant_peso(.50)},
                    {"medida": "Moda", "valor": moda},
                    {"medida": "Desviación estándar ponderada", "valor": np.sqrt(varianza)},
                    {"medida": "Q1 ponderado", "valor": cuant_peso(.25)},
                    {"medida": "Q3 ponderado", "valor": cuant_peso(.75)},
                    {"medida": "Personas representadas", "valor": total_personas}
                ])
                print("Tabla de edades agregada; se aplicaron frecuencias como pesos.")
                display(resumen_edad)
                guardar_resumen(resumen_edad, "estadisticas_edad_ponderadas.csv.gz")
            else:
                print("No hay frecuencias positivas para calcular edades ponderadas.")
        else:
            if len(edades):
                resumen_edad = pd.DataFrame([
                    {"medida": "Media", "valor": edades.mean()},
                    {"medida": "Mediana", "valor": edades.median()},
                    {"medida": "Moda(s)", "valor": ", ".join(map(str, edades.mode().tolist()))},
                    {"medida": "Desviación estándar", "valor": edades.std()},
                    {"medida": "Q1", "valor": edades.quantile(.25)},
                    {"medida": "Q3", "valor": edades.quantile(.75)},
                    {"medida": "N válido", "valor": len(edades)}
                ])
                display(resumen_edad)
                guardar_resumen(resumen_edad, "estadisticas_edad.csv.gz")
                plt.figure(figsize=(9, 4))
                plt.boxplot(edades, showmeans=True)
                plt.title("Caja y bigotes — edades válidas SSC")
                plt.ylabel("Edad en años")
                plt.tight_layout()
                plt.show()
            else:
                print("No hay edades válidas entre 0 y 99 años.")
    else:
        print(f"No se identificó la columna de edad en {nombre_edad}.")
else:
    print("No se encontró un archivo limpio de personas por edad.")

## 8. Resumen de ejecución e interpretación

Los resultados se guardan en la carpeta de análisis de esta ejecución. La interpretación final debe basarse en las tablas y gráficas que efectivamente se generen. Si falta la variable de lesionados, no se presenta el conteo de hechos como si fuera un conteo de personas lesionadas. Los valores atípicos se investigan, no se eliminan automáticamente.

In [ ]:
resumen_ejecucion = pd.DataFrame([
    {"elemento": "Ejecución SSC utilizada", "valor": str(SSC_RUN)},
    {"elemento": "Archivos limpios leídos", "valor": len(tablas_ssc)},
    {"elemento": "Tabla de hechos identificada", "valor": nombre_hechos if hechos is not None else "No identificada"},
    {"elemento": "Columna de lesionados", "valor": COL_LESIONADOS or "Pendiente de identificar"},
    {"elemento": "Columna de alcaldía", "valor": COL_ALCALDIA or "Pendiente de identificar"},
    {"elemento": "Columna de hora de ocurrencia", "valor": COL_HORA or "Pendiente de identificar"},
    {"elemento": "Carpeta de resultados", "valor": str(RUN)}
])
display(resumen_ejecucion)
guardar_resumen(resumen_ejecucion, "resumen_ejecucion_analisis.csv.gz")
print("Análisis terminado. Resultados guardados en:", RUN)